python scripts/reformat_hlagyn.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_hlagyn.tsv

In [4]:
import pandas as pd
import os

def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [5]:
df_combined = pd.read_csv('../combined_hlagyn.tsv', sep='\t')

In [3]:
BASE_PATH = '../data/HLAGyn/'
dfs = []

dfs_para_influ = []
dfs_ph4 = []
dfs_covid = []
dfts_ctn = []

for file in os.listdir(BASE_PATH):
    if file.endswith(('.xls', '.xlsx')):
        print(file)
        df = load_table(BASE_PATH + file).assign(filename=file)
        dfs.append(df)

        columns = ''.join(df.columns.tolist())
        h1n1_column = 'H1N1' in columns or 'Influenza' in columns
        ctn_column = 'CT_N' in columns

        if h1n1_column:
            if 'Parainfluenza' in columns:
                dfs_para_influ.append(df)
            elif 'PH4' in columns:
                dfs_ph4.append(df)
            elif 'SARS-CoV-2' in columns:
                dfs_covid.append(df)
        elif ctn_column:
            dfts_ctn.append(df)
        else:
            print('Unknown type')

20220329_Painel Viral 24_Segunda quinzena de Março de 2022.xlsx
20220912_PR24_SE36.xlsx
20230130_COVID_SE04.xlsx
20221121_PR24_SE46.xlsx
20230320_PR4_SE11.xlsx
20230102_PR4_SE52.xlsx
20230116_PR4_SE02.xlsx
20221210_COVID_SE49.xlsx
20220926_PR4_SE38.xlsx
20230605_PR24_SE22.xlsx
20230808_PR24_SE31.xlsx
20220614_PR4_se23.xlsx
20220627_PR4_SE25.xlsx
20220912_Covid_SE36.xlsx
20230320_COVID_SE11.xlsx
20230130_PR4_SE04.xlsx
20230731_PR24_SE30.xlsx
20220823_PR4_SE33.xlsx
20230626_PR4_SE25.xlsx
20220601_PR4_se21.xlsx
20220808_PR24_SE31.xlsx
20220510_PR24_se18.xlsx
20230313_PR4_SE10.xlsx
20230403_PR4_SE13.xlsx
20230130_PR24_SE04.xlsx
20220912_PR4_SE36.xlsx
20230208_PR24_SE05.xlsx
20230717_PR4_SE28.xlsx
20220504_Painel Viral 24_SE 17_2022.xlsx
20230102_COVID_SE52.xlsx
20230529_PR24_SE21.xlsx
20230701_PR24_SE26.xlsx
20230213_PR4_SE06.xlsx
20230307_PR24_SE09.2023.xlsx
20230307_PR4_SE09.2023.xlsx
20220823_PR24_SE33.xlsx
20221121_COVID_SE46.xlsx
20220704_PR4_SE26.xlsx
20220919_PR24_SE37.xlsx
20221205

In [4]:
df_hla_flu = pd.concat(dfs_para_influ)
df_hla_ph4 = pd.concat(dfs_ph4)
df_hla_covid = pd.concat(dfs_covid)
df_hla_ctn = pd.concat(dfts_ctn)

## Duplicatas

In [6]:
df_duplicates = (
    df_combined
    .assign( one=1 )
    .groupby(['sample_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

0 nan nan


,num_duplicates
sample_id,


In [8]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

6 1 1


,num_duplicates
test_id,
2199883,1
2199884,1
2199885,1
2199886,1
2200182,1
2200778,1


In [9]:
df_os_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)


In [10]:
df_os_multiple_tests.head(10)

,kits
test_id,
2199883,"test_4,test_4"
2199884,"test_4,test_4"
2199885,"test_4,test_4"
2199886,"test_4,test_4"
2200182,"test_4,test_4"
2200778,"test_24,test_24"


In [11]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [12]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['test_24', 'test_4', 'covid_pcr'], dtype=object)

In [13]:
df_combined['state'].unique()

array(['Goiás', 'Bahia', nan, 'Distrito Federal'], dtype=object)

In [14]:
df_combined['location'].unique()

array(['GOIANIA', 'APARECIDA DE GOIANIA', 'RIO VERDE', 'NIQUELANDIA',
       'PALMEIRAS DE GOIAS', 'CACHOEIRA ALTA', 'ANAPOLIS', 'ITUMBIARA',
       'SANTA HELENA DE GOIAS', nan, 'NOVA VENEZA', 'CRIXAS', 'INHUMAS',
       'MINEIROS', 'CATALAO', 'PIRES DO RIO', 'BELA VISTA DE GOIAS',
       'ACREUNA', 'SENADOR CANEDO', 'CERES', 'IPAMERI', 'UIBAI', 'JATAI',
       'GOIANESIA', 'BRASILIA'], dtype=object)

In [15]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-08-19', '2002-01-01')

In [17]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 59,  92,  49,  73,  28,   9,  39,  25,   0,  80,  32,  12,  43,
         46,  20,  35,  62,   3,  13,  82,  19,  54,  57,  21,  78,  77,
         83,  96,  40,  36,  85,   1,  64,  55,  31,  14,  44,  29,  72,
         27,  53,  18,  69,  58,   8,  56,  51,  41,  75,  37,  76,  61,
         48,  11,  22,  70,  26,  42,  38,  45,  79,  67,  47,   2,  90,
         63,  30,  34,  60,  24,  86,  33,  74,  17,  50,  23,   4,   5,
          7,  66,   6,  10,  94,  52,  15,  81,  68,  84,  88,  65,  16,
         87,  89,  71,  97,  91,  98,  93,  -1, 101,  95,  99, 106, 104,
        107, 100, 112, 103, 111, 102, 113, 105]),
 -1,
 113)

In [18]:
df_combined['sex'].unique()

array(['M', 'F', 'I'], dtype=object)

In [19]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['Pos' 'Neg' 'NT']
FLUB_test_result ['Neg' 'Pos' 'NT']
VSR_test_result ['Neg' 'Pos' 'NT']
SC2_test_result ['Neg' 'Pos']
META_test_result ['Neg' 'NT' 'Pos']
RINO_test_result ['Pos' 'Neg' 'NT']
PARA_test_result ['Neg' 'Pos' 'NT']
ADENO_test_result ['Neg' 'Pos' 'NT']
BOCA_test_result ['Neg' 'Pos' 'NT']
COVS_test_result ['Neg' 'Pos' 'NT']
ENTERO_test_result ['Neg' 'Pos' 'NT']
BAC_test_result ['Neg' 'NT' 'Pos']


In [20]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [18]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [19]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [20]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())
        # Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['Pos' 'Neg' 'Neg,Neg' 'NT']
FLUB_test_result_kit ['Neg' 'Pos' 'Neg,Neg' 'NT']
VSR_test_result_kit ['Neg' 'Pos' 'Neg,Neg' 'Pos,Pos' 'NT']
SC2_test_result_kit ['Neg' 'Pos' 'Neg,Neg']
META_test_result_kit ['Neg' 'NT' 'Pos' 'NT,NT' 'Neg,Neg']
RINO_test_result_kit ['Pos' 'Neg' 'NT' 'NT,NT' 'Neg,Neg']
PARA_test_result_kit ['Neg' 'Pos' 'NT' 'NT,NT' 'Neg,Neg']
ADENO_test_result_kit ['Neg' 'Pos' 'NT' 'NT,NT' 'Neg,Neg']
BOCA_test_result_kit ['Neg' 'Pos' 'NT' 'NT,NT' 'Neg,Neg']
COVS_test_result_kit ['Neg' 'Pos' 'NT' 'NT,NT' 'Neg,Neg']
ENTERO_test_result_kit ['Neg' 'Pos' 'NT' 'NT,NT' 'Neg,Neg']
BAC_test_result_kit ['Neg' 'NT' 'NT,NT' 'Neg,Neg' 'Pos']


In [21]:
df_os_non_contraditory_test_results.query("SC2_test_result_kit == 'Neg,Neg'")

,,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit
test_id,test_kit,,,,,,,,,,,,
2199883,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
2199884,test_4,"Neg,Neg","Neg,Neg","Pos,Pos","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
2199885,test_4,"Neg,Neg","Neg,Neg","Pos,Pos","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
2199886,test_4,"Neg,Neg","Neg,Neg","Pos,Pos","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
2200182,test_4,"Neg,Neg","Neg,Neg","Pos,Pos","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
2200778,test_24,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg"


In [29]:
df_combined.query("test_id == 2200778")

,lab_id,test_id,test_kit,patient_id,sample_id,state,location,date_testing,epiweek,age,...,Ct_RDRP,geneS_detection,META_test_result,RINO_test_result,PARA_test_result,ADENO_test_result,BOCA_test_result,COVS_test_result,ENTERO_test_result,BAC_test_result
1681,HLAGyn,2200778,test_24,NaN,dee693bd80bacec9,Goiás,GOIANIA,2022-03-28,2022-04-02,5,...,NaN,NaN,Neg,Neg,Neg,Neg,Neg,Neg,Neg,Neg
1682,HLAGyn,2200778,test_24,NaN,b897be7483e76e1f,Goiás,APARECIDA DE GOIANIA,2022-03-28,2022-04-02,5,...,NaN,NaN,Neg,Neg,Neg,Neg,Neg,Neg,Neg,Neg


### Ids

Verificando se todos os Ids originais etão presentes no arquivo final

In [ ]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_full_raw['CODIGO REQUISICAO'])

In [ ]:
# Testar se todos os test_ids da planilha original estão na planilha final
# deve retornar um set vazio
set_test_ids_raw_data.difference(set_test_ids_final_df)

{'1200085178_100',
 '1410231927_100',
 '1421499943_100',
 '1421512238_100',
 '1421512252_100',
 '1421524508_100',
 '1421542791_100',
 '1421567484_600',
 '1421573025_900',
 '1421574120_100',
 '1421580321_100',
 '1421580345_100',
 '1421580359_600',
 '1421580372_100',
 '1421580414_700',
 '1421590360_100',
 '1421608991_700',
 '1421618661_100',
 '1421623648_100',
 '1421627267_100',
 '1421641149_200',
 '1421661054_100',
 '1421747497_400',
 '1450064023_100',
 '1450064862_100',
 '1600683873_100',
 '1600708233_100',
 '1600709104_100',
 '1600712587_100',
 '1650441468_100',
 '1650447361_100',
 '1650451652_100',
 '1660378198_100',
 '1720397744_100',
 '1860281386_100',
 '1870163611_100',
 '2230148082_100',
 '2240159856_100',
 '2240169073_100',
 '2240170521_100',
 '2240193197_100',
 '2240197205_100',
 '2310036024_100',
 '2320523931_100',
 '2320536507_100',
 '2320542723_100',
 '2320550391_100',
 '2320551123_100',
 '2320551273_100',
 '2320562210_100',
 '2320582241_100',
 '2350037816_100',
 '2350041791

In [ ]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

set()

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [32]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'flu_antigen':{
        'FLUA_test_result',
        'FLUB_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [33]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [34]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, RINO_test_result, FLUA_test_result, SC2_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, ENTERO_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, BOCA_test_result, RINO_test_result, FLUA_test_result]



flu_antigen
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result]



covid_antigen
Empty DataFrame
Columns: []
Index: [SC2_test_result]



covid_pcr
                1929
SC2_test_result  Tes





Verificando resultados de alguns testes

In [35]:
import numpy as np

In [51]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [52]:
current_id=50

In [78]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith('Vírus')]].T )
print( df_hla_ctn.query(f"Pedido == '{id}'") [['Pedido', 'Resultado']].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                         7665
test_id               2262865
test_kit            covid_pcr
FLUA_test_result           NT
FLUB_test_result           NT
VSR_test_result            NT
SC2_test_result           Neg
META_test_result           NT
RINO_test_result           NT
PARA_test_result           NT
ADENO_test_result          NT
BOCA_test_result           NT
COVS_test_result           NT
ENTERO_test_result         NT
BAC_test_result            NT
Empty DataFrame
Columns: []
Index: [Pedido, Vírus Influenza A, Vírus Influenza B, Vírus Sincicial Respiratório A/B]
                   1912
Pedido          2262865
Resultado  Inconclusivo
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4, VIRUS_Adenovirus, VIRUS_Bocavirus, VIRUS_CoV-229E, VIRUS_CoV-HKU, VI

Verificando resultados de alguns testes - Fltrando por test_kit

In [84]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [85]:
current_id=50

In [107]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith(('Vírus', 'Coron'))]].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                      50943
test_id             2354190
test_kit             test_4
FLUA_test_result        Neg
FLUB_test_result        Neg
VSR_test_result         Pos
SC2_test_result         Neg
META_test_result         NT
RINO_test_result         NT
PARA_test_result         NT
ADENO_test_result        NT
BOCA_test_result         NT
COVS_test_result         NT
ENTERO_test_result       NT
BAC_test_result          NT
                                             18
Pedido                                  2354190
Vírus Influenza A                 Não Detectado
Vírus Influenza B                 Não Detectado
Vírus Sincicial Respiratório A/B      Detectado
Coronavírus SARS-CoV-2            Não Detectado
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4